## Why the container filesystem isn't enough

Recall the layer stack from module 02: read-only image layers with one thin **writable layer** on top, where a running container's changes land.

```
  writable container layer   ← ephemeral: deleted with the container
  read-only image layers
```

When a container's process writes to its own filesystem — a Postgres container writing to `/var/lib/postgresql/data` — those bytes go into that writable layer. It works, but it has **three problems** that make it wrong for real data:

1. **It's ephemeral.** `docker rm` deletes the container and takes the writable layer with it. Your database is gone. (This is the module-02 point that the writable layer dies with the container — fine for scratch, fatal for data.)
2. **It's not shareable.** Two containers can't see each other's writable layers, so you can't have a sidecar reading another container's files.
3. **It's slow for heavy writes.** overlay2 does a **copy-up** on first write of each file — fine for the odd config change, expensive for write-heavy workloads like databases.

The fix is one idea: **take a path inside the container and mount something else there** — storage that lives *outside* the writable layer and *survives* the container's lifecycle. Mount a volume at `/var/lib/postgresql/data` and now `docker rm` the database container, start a fresh one with the same mount, and the data is still there.

That single move — mount external storage over a container path — is what the whole module is built on. The only remaining question is *what kind* of storage you mount, which is the next section: volumes, bind mounts, or `tmpfs`.